In [ ]:
import boto3, botocore

PROFILE = "whizdomai-platform-prod"      
BUCKET = "whizdomai-datalake"
PREFIX = "gamingtec/datalake/00-reference-data/whizdomai-games/"

session = boto3.Session(profile_name=PROFILE)
print("identity:", session.client("sts").get_caller_identity()["Arn"])   

s3 = session.client("s3")
try:
    r = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, MaxKeys=10)
    print("keys:", r.get("KeyCount"), [o["Key"] for o in r.get("Contents", [])][:5])
except botocore.exceptions.ClientError as e:
    print(e.response["Error"]["Code"], e.response["Error"]["Message"])   


identity: arn:aws:sts::439389909314:assumed-role/AWSReservedSSO_WhizdomaiDataScientistReadOnly_d0bef6c8f79424fb/javier.herrero
keys: 10 ['gamingtec/datalake/00-reference-data/whizdomai-games/', 'gamingtec/datalake/00-reference-data/whizdomai-games/README.md', 'gamingtec/datalake/00-reference-data/whizdomai-games/config/README.md', 'gamingtec/datalake/00-reference-data/whizdomai-games/config/__init__.py', 'gamingtec/datalake/00-reference-data/whizdomai-games/config/__pycache__/__init__.cpython-312.pyc']


In [4]:
readme = next((o["Key"] for o in r.get("Contents", []) if o["Key"].lower().endswith("readme.md")), None)
print("README:", readme)

if readme:
    text = s3.get_object(Bucket=BUCKET, Key=readme)["Body"].read().decode("utf-8")
    print(text)


README: gamingtec/datalake/00-reference-data/whizdomai-games/README.md
# SlotCatalog Game Extraction Pipeline

Automated pipeline to extract game metadata from SlotCatalog.com and map it to our internal game taxonomy schema using AI (Claude API or local Ollama models).

## Features

- **Stage 1: Web Scraper** - Extracts raw game data from SlotCatalog pages
- **Stage 2: Taxonomy Mapper** - Uses AI to intelligently map raw data to taxonomy schema
  - **Claude API** (paid) - High accuracy, fast
  - **Ollama** (free) - Local open-source models, no API costs
- **Intelligent Mapping** - Automatic theme detection, feature extraction, and data normalization
- **Rate Limiting** - Respectful scraping with configurable delays
- **Error Handling** - Robust retry logic and comprehensive error reporting
- **Resumable** - Can resume from interruptions

## Quick Start

### 1. Installation

```bash
# Create virtual environment
python -m venv venv

# Activate virtual environment
source venv/bin/activate

In [5]:
from collections import Counter

paginator = s3.get_paginator("list_objects_v2")
objects = [o for page in paginator.paginate(Bucket=BUCKET, Prefix=PREFIX) for o in page.get("Contents", [])]
print("Total de objetos:", len(objects))

ext = Counter(o["Key"].rsplit(".", 1)[-1].lower() if "." in o["Key"].split("/")[-1] else "(sin extensión)" for o in objects)
print("Por extensión:", dict(ext.most_common()))

print("\nZip y JSON (los 20 primeros):")
for o in [o for o in objects if o["Key"].lower().endswith((".zip", ".json"))][:20]:
    print(f"  {o['Key'].removeprefix(PREFIX):60s} {o['Size']:>14,} bytes")


Total de objetos: 50
Por extensión: {'py': 18, 'md': 10, 'pyc': 7, 'txt': 4, 'json': 4, 'ipynb': 2, '(sin extensión)': 1, 'zip': 1, 'xlsx': 1, 'html': 1, 'sh': 1}

Zip y JSON (los 20 primeros):
  config/mapping_template.json                                          1,765 bytes
  data.zip                                                         53,692,976 bytes
  reference/games_taxonomy.json                                        23,919 bytes
  reference/theme_mappin.json                                         157,808 bytes
  scripts/games.json                                                    3,450 bytes


In [6]:
import io, zipfile

zip_key = next(o["Key"] for o in objects if o["Key"].lower().endswith("data.zip"))
print("Zip:", zip_key)

# Descarga a memoria; si pesa varios cientos de MB, avísame antes de ejecutarlo
body = s3.get_object(Bucket=BUCKET, Key=zip_key)["Body"].read()
print(f"Tamaño: {len(body):,} bytes")

with zipfile.ZipFile(io.BytesIO(body)) as z:
    names = z.namelist()
    print("Ficheros dentro:", len(names))
    for n in names[:20]:
        print(f"  {n:70s} {z.getinfo(n).file_size:>12,} bytes")


Zip: gamingtec/datalake/00-reference-data/whizdomai-games/data.zip
Tamaño: 53,692,976 bytes
Ficheros dentro: 19180
  data/                                                                             0 bytes
  __MACOSX/._data                                                                 163 bytes
  data/.DS_Store                                                               20,484 bytes
  __MACOSX/data/._.DS_Store                                                       120 bytes
  data/mapping_summary.json                                                       262 bytes
  __MACOSX/data/._mapping_summary.json                                            219 bytes
  data/input/                                                                       0 bytes
  __MACOSX/data/._input                                                           163 bytes
  data/source/                                                                      0 bytes
  __MACOSX/data/._source                                 

In [7]:
# Celda 4: qué carpetas hay dentro del zip (ignorando la basura de macOS)
from collections import Counter

with zipfile.ZipFile(io.BytesIO(body)) as z:
    names = [n for n in z.namelist() if not n.startswith("__MACOSX") and not n.endswith(("/", ".DS_Store"))]
for folder, count in Counter("/".join(n.split("/")[:2]) for n in names).most_common():
    print(f"{count:6d}  {folder}")


  9533  data/raw
    13  data/input
    12  data/mapped_test
     6  data/gold
     4  data/source
     4  data/mapped
     3  data/final
     1  data/mapping_summary.json
     1  data/README.md
     1  data/output
     1  data/interim


In [8]:
# Celda 5: estructura de un JSON "raw" (claves y tipos), sin volcar todo el contenido
import json

raw_names = [n for n in names if n.startswith("data/raw/") and n.endswith(".json")]
print(len(raw_names), "ficheros raw; el primero:", raw_names[:1])

with zipfile.ZipFile(io.BytesIO(body)) as z:
    sample = json.loads(z.read(raw_names[0]))

def show(x, depth=0, max_depth=4):
    pad = "  " * depth
    if isinstance(x, dict) and depth < max_depth:
        for k, v in x.items():
            print(f"{pad}{k}: {type(v).__name__}" + (f" ({len(v)})" if isinstance(v, (list, dict)) else f" = {str(v)[:60]!r}"))
            show(v, depth + 1, max_depth)
    elif isinstance(x, list) and x and depth < max_depth:
        print(f"{pad}[0]: {type(x[0]).__name__}")
        show(x[0], depth + 1, max_depth)

show(sample)


9532 ficheros raw; el primero: ['data/raw/5419c8c4-bb73-43e2-8e53-e8a9c09f98e0.json']
game_id: str = '5419c8c4-bb73-43e2-8e53-e8a9c09f98e0'
source_url: str = 'https://slotcatalog.com/en/slots/MonkeyPop'
extraction_timestamp: str = '2025-11-28T07:11:28.004633+00:00'
raw_data: dict (17)
  title: str = 'Play MonkeyPop'
  provider: str = 'AvatarUX Studios'
  rtp_text: str = '96.54%'
  volatility_text: str = 'High'
  max_win_text: str = 'x5787.00'
  paylines_text: str = '486(Bothway)[ i ]'
  release_date_text: str = '25/04/2022[ i ]'
  game_type: str = 'Video Slots'
  hit_frequency: str = '22.7[ i ]'
  min_bet_text: str = '0.2'
  max_bet_text: str = '100'
  grid_text: str = '5-3[ i ]'
  features_text: str = 'Additional Free Spins,Bothway,Buy Feature,Free Spins,Free Sp'
  theme_text: str = 'Monkey'
  other_tags: str = '5 Reels,96% RTP'
  technology: str = 'JS, HTML5'
  description: str = 'SlotMonkeyPop is a medium volatile and stylish watercolor as'
extraction_status: str = 'success'
errors:

In [9]:
from pathlib import Path

dest = Path("/home/javier/whizdom-churn/catalog_project/internal_taxonomy/sa_raw")
dest.mkdir(exist_ok=True)

with zipfile.ZipFile(io.BytesIO(body)) as z:
    for n in z.namelist():
        if (n.startswith("data/raw/") and n.endswith(".json")) or n in ("data/README.md", "data/mapping_summary.json"):
            (dest / Path(n).name).write_bytes(z.read(n))

print(len(list(dest.glob("*.json"))), "ficheros en", dest)


9533 ficheros en /home/javier/whizdom-churn/catalog_project/internal_taxonomy/sa_raw


In [ ]:
from pathlib import Path

dest = Path("/home/javier/whizdom-churn/catalog_project/internal_taxonomy/sa_raw")
dest.mkdir(exist_ok=True)

with zipfile.ZipFile(io.BytesIO(body)) as z:
    for n in z.namelist():
        if (n.startswith("data/raw/") and n.endswith(".json")) or n in ("data/README.md", "data/mapping_summary.json"):
            (dest / Path(n).name).write_bytes(z.read(n))

print(len(list(dest.glob("*.json"))), "ficheros en", dest)
